# Behavior-Aware SASRec

Train and compare:
- **purchase_only**: SASRec on purchase sequences only
- **behavior_aware**: same SASRec backbone + cart add/remove behavior embeddings

Target is always the **next purchase**. Cart events are used as context before the target purchase.

This notebook is self-contained (no external package).

## 0. Setup

In [ ]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple
import json
import math

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

NOTEBOOK_DIR = Path.cwd()
# Prefer artifacts next to this notebook when cwd is the modeling folder.
if (NOTEBOOK_DIR / "Behavior_Aware_SASRec.ipynb").exists():
    ARTIFACT_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR / "behavior_aware_sasrec" / "Behavior_Aware_SASRec.ipynb").exists():
    ARTIFACT_ROOT = NOTEBOOK_DIR / "behavior_aware_sasrec"
else:
    ARTIFACT_ROOT = NOTEBOOK_DIR

print("Artifact root:", ARTIFACT_ROOT.resolve())
print("Torch:", torch.__version__, "| device:", "cuda" if torch.cuda.is_available() else "cpu")

## 1. Config

Toggle `USE_SYNTHETIC` for a quick dry-run without parquet files.  
For the real experiment, set `USE_SYNTHETIC = False` and point `DATA_DIR` at your Step 1 subsample.

In [ ]:
# --- data ---
USE_SYNTHETIC = True          # True: fake data smoke test; False: Synerise subsample
DATA_DIR = Path("data/subsampled")  # relative to repo root when USE_SYNTHETIC=False
TOP_N_ITEMS = 5000            # None = keep all items; global purchase-frequency pool
MIN_BUYS = 2

# --- model / train ---
MODE = "both"                 # "purchase_only" | "behavior_aware" | "both"
MAX_LEN = 50
HIDDEN_SIZE = 64
N_BLOCKS = 2
N_HEADS = 1
DROPOUT = 0.2
BATCH_SIZE = 128
LR = 1e-3
EPOCHS = 3 if USE_SYNTHETIC else 20
N_NEGATIVES = 1
EVAL_NEGATIVES = 100
PATIENCE = 5
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

OUTPUT_DIR = ARTIFACT_ROOT / "artifacts" / ("notebook_synthetic" if USE_SYNTHETIC else "notebook_run")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("USE_SYNTHETIC =", USE_SYNTHETIC)
print("DEVICE =", DEVICE)
print("OUTPUT_DIR =", OUTPUT_DIR)

## 2. Data helpers (sequences + temporal split)

In [ ]:
BEHAVIOR_PAD = 0
BEHAVIOR_BUY = 1
BEHAVIOR_ADD = 2
BEHAVIOR_REMOVE = 3

BEHAVIOR_NAME_TO_ID = {
    "product_buy": BEHAVIOR_BUY,
    "buy": BEHAVIOR_BUY,
    "purchase": BEHAVIOR_BUY,
    "add_to_cart": BEHAVIOR_ADD,
    "add": BEHAVIOR_ADD,
    "remove_from_cart": BEHAVIOR_REMOVE,
    "remove": BEHAVIOR_REMOVE,
}


@dataclass(frozen=True)
class BehaviorEvent:
    item_id: int
    behavior_id: int
    timestamp: np.datetime64


def load_behavior_events(
    data_dir: Path | str,
    include_behaviors: Sequence[str] = ("product_buy", "add_to_cart", "remove_from_cart"),
    file_map: Optional[Dict[str, str]] = None,
) -> pd.DataFrame:
    data_dir = Path(data_dir)
    default_files = {
        "product_buy": "product_buy_subsampled.parquet",
        "add_to_cart": "add_to_cart_subsampled.parquet",
        "remove_from_cart": "remove_from_cart_subsampled.parquet",
    }
    files = file_map or default_files

    frames: List[pd.DataFrame] = []
    for behavior in include_behaviors:
        path = data_dir / files[behavior]
        if not path.exists():
            raise FileNotFoundError(f"Missing event file for {behavior}: {path}")
        df = pd.read_parquet(path)
        df = df.drop_duplicates().copy()
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
        df = df.dropna(subset=["timestamp", "client_id", "sku"])
        df["behavior"] = behavior
        frames.append(df[["client_id", "sku", "timestamp", "behavior"]])

    events = pd.concat(frames, ignore_index=True)
    events = events.sort_values(["client_id", "timestamp", "behavior"]).reset_index(drop=True)
    return events


def _remap_ids(series: pd.Series) -> Tuple[pd.Series, Dict[int, int], Dict[int, int]]:
    uniques = pd.Index(sorted(series.unique()))
    raw_to_idx = {int(raw): i + 1 for i, raw in enumerate(uniques)}
    idx_to_raw = {v: k for k, v in raw_to_idx.items()}
    mapped = series.map(raw_to_idx).astype(np.int64)
    return mapped, raw_to_idx, idx_to_raw


def build_user_sequences(
    events: pd.DataFrame,
    include_behaviors: Sequence[str] = ("product_buy", "add_to_cart", "remove_from_cart"),
    min_buys: int = 2,
    max_len: Optional[int] = None,
    top_n_items: Optional[int] = None,
) -> Dict:
    df = events[events["behavior"].isin(list(include_behaviors))].copy()
    if df.empty:
        raise ValueError("No events left after filtering behaviors.")

    if top_n_items is not None:
        top_items = (
            df.loc[df["behavior"] == "product_buy", "sku"]
            .value_counts()
            .head(top_n_items)
            .index
        )
        buy_mask = (df["behavior"] == "product_buy") & df["sku"].isin(top_items)
        cart_mask = (df["behavior"] != "product_buy") & df["sku"].isin(top_items)
        df = df.loc[buy_mask | cart_mask].copy()

    buys_per_user = df.loc[df["behavior"] == "product_buy"].groupby("client_id").size()
    keep_users = set(buys_per_user[buys_per_user >= min_buys].index)
    df = df[df["client_id"].isin(keep_users)].copy()
    if df.empty:
        raise ValueError("No users left after min_buys filter.")

    df["item_id"], item_raw_to_idx, item_idx_to_raw = _remap_ids(df["sku"])
    df["user_id"], user_raw_to_idx, user_idx_to_raw = _remap_ids(df["client_id"])
    df["behavior_id"] = df["behavior"].map(BEHAVIOR_NAME_TO_ID).astype(np.int64)

    sequences: Dict[int, List[BehaviorEvent]] = {}
    for user_id, group in df.groupby("user_id", sort=True):
        group = group.sort_values("timestamp")
        events_list = [
            BehaviorEvent(
                item_id=int(row.item_id),
                behavior_id=int(row.behavior_id),
                timestamp=np.datetime64(row.timestamp.to_datetime64()),
            )
            for row in group.itertuples(index=False)
        ]
        if max_len is not None and len(events_list) > max_len:
            events_list = events_list[-max_len:]
        sequences[int(user_id)] = events_list

    return {
        "sequences": sequences,
        "n_items": len(item_raw_to_idx),
        "n_users": len(user_raw_to_idx),
        "item_raw_to_idx": item_raw_to_idx,
        "item_idx_to_raw": item_idx_to_raw,
        "user_raw_to_idx": user_raw_to_idx,
        "user_idx_to_raw": user_idx_to_raw,
        "include_behaviors": list(include_behaviors),
    }


def temporal_split_sequences(
    sequences: Dict[int, List[BehaviorEvent]],
    train_ratio: float = 0.7,
    val_ratio: float = 0.15,
) -> Dict[str, Dict[int, dict]]:
    """Leave-one-purchase-out: last buy=test, second-to-last=val; history is events before target."""
    if train_ratio <= 0 or val_ratio < 0 or train_ratio + val_ratio >= 1:
        raise ValueError("Require train_ratio > 0, val_ratio >= 0, train+val < 1.")

    splits = {"train": {}, "val": {}, "test": {}}

    for user_id, seq in sequences.items():
        buy_idx = [i for i, e in enumerate(seq) if e.behavior_id == BEHAVIOR_BUY]
        if len(buy_idx) < 2:
            continue

        test_pos = buy_idx[-1]
        test_hist = seq[:test_pos]
        test_target = seq[test_pos]
        if not test_hist:
            continue
        splits["test"][user_id] = {
            "history": test_hist,
            "target_item": test_target.item_id,
            "target_time": test_target.timestamp,
        }

        if len(buy_idx) >= 3:
            val_pos = buy_idx[-2]
            val_hist = seq[:val_pos]
            val_target = seq[val_pos]
            if val_hist:
                splits["val"][user_id] = {
                    "history": val_hist,
                    "target_item": val_target.item_id,
                    "target_time": val_target.timestamp,
                }
            train_hist = seq[:val_pos]
        else:
            train_hist = test_hist

        train_buy_idx = [i for i, e in enumerate(train_hist) if e.behavior_id == BEHAVIOR_BUY]
        if len(train_buy_idx) < 1:
            continue

        instances = []
        for bpos in train_buy_idx:
            if bpos == 0:
                continue
            instances.append(
                {
                    "history": train_hist[:bpos],
                    "target_item": train_hist[bpos].item_id,
                    "target_time": train_hist[bpos].timestamp,
                }
            )
        if instances:
            splits["train"][user_id] = instances

    return splits


def filter_purchase_only(seq: Sequence[BehaviorEvent]) -> List[BehaviorEvent]:
    return [e for e in seq if e.behavior_id == BEHAVIOR_BUY]


def make_synthetic_events(
    n_users: int = 200,
    n_items: int = 100,
    seed: int = 42,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    start = np.datetime64("2022-06-01")
    rows = []
    for u in range(1, n_users + 1):
        n_buys = int(rng.integers(3, 8))
        t = start + rng.integers(0, 10).astype("timedelta64[D]")
        for _ in range(n_buys):
            item = int(rng.integers(1, n_items + 1))
            if rng.random() < 0.8:
                t = t + rng.integers(1, 5).astype("timedelta64[h]")
                rows.append((u, item, pd.Timestamp(t), "add_to_cart"))
                if rng.random() < 0.15:
                    t = t + rng.integers(1, 3).astype("timedelta64[h]")
                    rows.append((u, item, pd.Timestamp(t), "remove_from_cart"))
                    if rng.random() < 0.5:
                        t = t + rng.integers(1, 5).astype("timedelta64[h]")
                        rows.append((u, item, pd.Timestamp(t), "add_to_cart"))
            t = t + rng.integers(1, 12).astype("timedelta64[h]")
            rows.append((u, item, pd.Timestamp(t), "product_buy"))
            if rng.random() < 0.3:
                other = int(rng.integers(1, n_items + 1))
                t = t + rng.integers(1, 6).astype("timedelta64[h]")
                rows.append((u, other, pd.Timestamp(t), "add_to_cart"))
    return pd.DataFrame(rows, columns=["client_id", "sku", "timestamp", "behavior"])

print("Data helpers ready")

## 3. Dataset + metrics + models + train loop

In [ ]:
def _pad_or_truncate(
    item_ids: List[int],
    behavior_ids: List[int],
    max_len: int,
) -> tuple[List[int], List[int]]:
    if len(item_ids) > max_len:
        item_ids = item_ids[-max_len:]
        behavior_ids = behavior_ids[-max_len:]
    pad = max_len - len(item_ids)
    if pad > 0:
        item_ids = [0] * pad + item_ids
        behavior_ids = [BEHAVIOR_PAD] * pad + behavior_ids
    return item_ids, behavior_ids


class SASRecDataset(Dataset):
    def __init__(
        self,
        split_data: Dict[int, object],
        max_len: int = 50,
        purchase_only: bool = False,
        n_items: Optional[int] = None,
        n_negatives: int = 1,
        seed: int = 42,
    ) -> None:
        self.max_len = max_len
        self.purchase_only = purchase_only
        self.n_items = n_items
        self.n_negatives = n_negatives
        self.rng = np.random.default_rng(seed)
        self.examples: List[dict] = []

        for user_id, payload in split_data.items():
            instances = payload if isinstance(payload, list) else [payload]
            for inst in instances:
                history: Sequence[BehaviorEvent] = inst["history"]
                if purchase_only:
                    history = filter_purchase_only(history)
                if not history:
                    continue
                self.examples.append(
                    {
                        "user_id": int(user_id),
                        "history": list(history),
                        "target_item": int(inst["target_item"]),
                    }
                )

    def __len__(self) -> int:
        return len(self.examples)

    def _sample_negatives(self, positive: int, banned: set[int]) -> List[int]:
        assert self.n_items is not None
        negs: List[int] = []
        while len(negs) < self.n_negatives:
            neg = int(self.rng.integers(1, self.n_items + 1))
            if neg != positive and neg not in banned:
                negs.append(neg)
        return negs

    def __getitem__(self, idx: int) -> dict:
        ex = self.examples[idx]
        items = [e.item_id for e in ex["history"]]
        behaviors = [e.behavior_id for e in ex["history"]]
        items, behaviors = _pad_or_truncate(items, behaviors, self.max_len)

        out = {
            "user_id": ex["user_id"],
            "item_ids": torch.tensor(items, dtype=torch.long),
            "behavior_ids": torch.tensor(behaviors, dtype=torch.long),
            "target_item": torch.tensor(ex["target_item"], dtype=torch.long),
        }
        if self.n_negatives > 0 and self.n_items is not None:
            banned = {e.item_id for e in ex["history"] if e.behavior_id == BEHAVIOR_BUY}
            banned.add(ex["target_item"])
            negs = self._sample_negatives(ex["target_item"], banned)
            out["neg_items"] = torch.tensor(negs, dtype=torch.long)
        return out


def collate_sasrec(batch: List[dict]) -> dict:
    collated = {
        "user_id": torch.tensor([b["user_id"] for b in batch], dtype=torch.long),
        "item_ids": torch.stack([b["item_ids"] for b in batch], dim=0),
        "behavior_ids": torch.stack([b["behavior_ids"] for b in batch], dim=0),
        "target_item": torch.stack([b["target_item"] for b in batch], dim=0),
    }
    if "neg_items" in batch[0]:
        collated["neg_items"] = torch.stack([b["neg_items"] for b in batch], dim=0)
    return collated


def hit_rate_at_k(ranks: np.ndarray, k: int) -> float:
    return float((ranks <= k).mean()) if len(ranks) else 0.0


def ndcg_at_k(ranks: np.ndarray, k: int) -> float:
    if len(ranks) == 0:
        return 0.0
    gains = np.where(ranks <= k, 1.0 / np.log2(ranks.astype(np.float64) + 1.0), 0.0)
    return float(gains.mean())


def mrr(ranks: np.ndarray) -> float:
    return float((1.0 / ranks).mean()) if len(ranks) else 0.0


@torch.no_grad()
def rank_targets(scores: torch.Tensor, target_items: torch.Tensor) -> np.ndarray:
    target_scores = scores.gather(1, target_items.view(-1, 1)).squeeze(1)
    better = scores[:, 1:] > target_scores.unsqueeze(1)
    ranks = better.sum(dim=1) + 1
    return ranks.cpu().numpy().astype(np.int64)


@torch.no_grad()
def rank_targets_among_candidates(cand_scores: torch.Tensor, pos_index: torch.Tensor) -> np.ndarray:
    pos_scores = cand_scores.gather(1, pos_index.view(-1, 1)).squeeze(1)
    better = cand_scores > pos_scores.unsqueeze(1)
    ranks = better.sum(dim=1) + 1
    return ranks.cpu().numpy().astype(np.int64)


def summarize_ranks(ranks: np.ndarray, ks: Iterable[int] = (5, 10)) -> Dict[str, float]:
    out: Dict[str, float] = {"MRR": mrr(ranks), "n": float(len(ranks))}
    for k in ks:
        out[f"HR@{k}"] = hit_rate_at_k(ranks, k)
        out[f"NDCG@{k}"] = ndcg_at_k(ranks, k)
    return out


def format_metrics(metrics: Dict[str, float]) -> str:
    parts: List[str] = []
    for key in sorted(metrics.keys()):
        if key == "n":
            parts.append(f"n={int(metrics[key])}")
        else:
            parts.append(f"{key}={metrics[key]:.4f}")
    return " | ".join(parts)


class PointWiseFeedForward(nn.Module):
    def __init__(self, hidden_size: int, dropout: float) -> None:
        super().__init__()
        self.conv1 = nn.Conv1d(hidden_size, hidden_size, kernel_size=1)
        self.dropout1 = nn.Dropout(dropout)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv1d(hidden_size, hidden_size, kernel_size=1)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = x.transpose(1, 2)
        out = self.dropout2(self.conv2(self.relu(self.dropout1(self.conv1(out)))))
        return out.transpose(1, 2)


class SASRecBlock(nn.Module):
    def __init__(self, hidden_size: int, n_heads: int, dropout: float) -> None:
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_size, num_heads=n_heads, dropout=dropout, batch_first=True
        )
        self.ffn = PointWiseFeedForward(hidden_size, dropout)
        self.norm1 = nn.LayerNorm(hidden_size, eps=1e-8)
        self.norm2 = nn.LayerNorm(hidden_size, eps=1e-8)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        attn_mask: torch.Tensor,
        key_padding_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        h = self.norm1(x)
        attn_out, _ = self.attn(
            h, h, h, attn_mask=attn_mask, key_padding_mask=key_padding_mask, need_weights=False
        )
        x = x + self.dropout(attn_out)
        h = self.norm2(x)
        return x + self.ffn(h)


class SASRec(nn.Module):
    def __init__(
        self,
        n_items: int,
        hidden_size: int = 64,
        max_len: int = 50,
        n_blocks: int = 2,
        n_heads: int = 1,
        dropout: float = 0.2,
    ) -> None:
        super().__init__()
        self.n_items = n_items
        self.hidden_size = hidden_size
        self.max_len = max_len
        self.item_emb = nn.Embedding(n_items + 1, hidden_size, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, hidden_size)
        self.emb_dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [SASRecBlock(hidden_size, n_heads, dropout) for _ in range(n_blocks)]
        )
        self.final_norm = nn.LayerNorm(hidden_size, eps=1e-8)
        self._init_weights()

    def _init_weights(self) -> None:
        std = 0.02
        for module in self.modules():
            if isinstance(module, (nn.Linear, nn.Embedding, nn.Conv1d)):
                nn.init.normal_(module.weight, mean=0.0, std=std)
            if isinstance(module, nn.Embedding) and module.padding_idx is not None:
                with torch.no_grad():
                    module.weight[module.padding_idx].fill_(0)
            if isinstance(module, nn.LayerNorm):
                nn.init.zeros_(module.bias)
                nn.init.ones_(module.weight)

    def _causal_mask(self, seq_len: int, device: torch.device) -> torch.Tensor:
        return torch.triu(torch.ones(seq_len, seq_len, device=device, dtype=torch.bool), diagonal=1)

    def encode(
        self,
        item_ids: torch.Tensor,
        behavior_ids: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        del behavior_ids
        batch_size, seq_len = item_ids.shape
        positions = torch.arange(seq_len, device=item_ids.device).unsqueeze(0).expand(batch_size, -1)
        x = self.item_emb(item_ids) + self.pos_emb(positions)
        x = self.emb_dropout(x)
        pad_mask = item_ids == 0
        attn_mask = self._causal_mask(seq_len, item_ids.device)
        for block in self.blocks:
            x = block(x, attn_mask=attn_mask, key_padding_mask=pad_mask)
        return self.final_norm(x)

    def forward(
        self,
        item_ids: torch.Tensor,
        behavior_ids: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        seq_emb = self.encode(item_ids, behavior_ids)
        return torch.matmul(seq_emb, self.item_emb.weight.transpose(0, 1))

    def predict(
        self,
        item_ids: torch.Tensor,
        behavior_ids: Optional[torch.Tensor] = None,
        candidate_items: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        seq_emb = self.encode(item_ids, behavior_ids)
        lengths = (item_ids != 0).sum(dim=1).clamp(min=1)
        last_idx = lengths - 1
        final = seq_emb[torch.arange(seq_emb.size(0), device=seq_emb.device), last_idx]
        if candidate_items is None:
            return torch.matmul(final, self.item_emb.weight.transpose(0, 1))
        cand_emb = self.item_emb(candidate_items)
        return (cand_emb * final.unsqueeze(1)).sum(dim=-1)


class BehaviorAwareSASRec(SASRec):
    def __init__(
        self,
        n_items: int,
        n_behaviors: int = 3,
        hidden_size: int = 64,
        max_len: int = 50,
        n_blocks: int = 2,
        n_heads: int = 1,
        dropout: float = 0.2,
    ) -> None:
        super().__init__(
            n_items=n_items,
            hidden_size=hidden_size,
            max_len=max_len,
            n_blocks=n_blocks,
            n_heads=n_heads,
            dropout=dropout,
        )
        self.behavior_emb = nn.Embedding(n_behaviors + 1, hidden_size, padding_idx=0)
        nn.init.normal_(self.behavior_emb.weight, mean=0.0, std=0.02)
        with torch.no_grad():
            self.behavior_emb.weight[0].fill_(0)

    def encode(
        self,
        item_ids: torch.Tensor,
        behavior_ids: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        if behavior_ids is None:
            raise ValueError("BehaviorAwareSASRec requires behavior_ids.")
        batch_size, seq_len = item_ids.shape
        positions = torch.arange(seq_len, device=item_ids.device).unsqueeze(0).expand(batch_size, -1)
        x = self.item_emb(item_ids) + self.behavior_emb(behavior_ids) + self.pos_emb(positions)
        x = self.emb_dropout(x)
        pad_mask = item_ids == 0
        attn_mask = self._causal_mask(seq_len, item_ids.device)
        for block in self.blocks:
            x = block(x, attn_mask=attn_mask, key_padding_mask=pad_mask)
        return self.final_norm(x)


def bce_ranking_loss(
    model: SASRec,
    item_ids: torch.Tensor,
    behavior_ids: torch.Tensor,
    pos_items: torch.Tensor,
    neg_items: torch.Tensor,
) -> torch.Tensor:
    seq_emb = model.encode(item_ids, behavior_ids)
    lengths = (item_ids != 0).sum(dim=1).clamp(min=1)
    last_idx = lengths - 1
    final = seq_emb[torch.arange(seq_emb.size(0), device=seq_emb.device), last_idx]
    pos_emb = model.item_emb(pos_items)
    pos_logits = (final * pos_emb).sum(dim=-1)
    neg_emb = model.item_emb(neg_items)
    if neg_emb.dim() == 2:
        neg_logits = (final * neg_emb).sum(dim=-1)
        return -F.logsigmoid(pos_logits).mean() - F.logsigmoid(-neg_logits).mean()
    neg_logits = (neg_emb * final.unsqueeze(1)).sum(dim=-1)
    return -F.logsigmoid(pos_logits).mean() - F.logsigmoid(-neg_logits).mean()


@dataclass
class TrainConfig:
    hidden_size: int = 64
    max_len: int = 50
    n_blocks: int = 2
    n_heads: int = 1
    dropout: float = 0.2
    batch_size: int = 128
    lr: float = 1e-3
    weight_decay: float = 0.0
    epochs: int = 20
    n_negatives: int = 1
    eval_negatives: int = 100
    num_workers: int = 0
    seed: int = 42
    device: str = "cpu"
    patience: int = 5


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)


def build_model(mode: str, n_items: int, cfg: TrainConfig) -> SASRec:
    common = dict(
        n_items=n_items,
        hidden_size=cfg.hidden_size,
        max_len=cfg.max_len,
        n_blocks=cfg.n_blocks,
        n_heads=cfg.n_heads,
        dropout=cfg.dropout,
    )
    if mode == "purchase_only":
        return SASRec(**common)
    if mode == "behavior_aware":
        return BehaviorAwareSASRec(n_behaviors=3, **common)
    raise ValueError(f"Unknown mode: {mode}")


@torch.no_grad()
def evaluate(
    model: SASRec,
    split_data: Dict[int, object],
    n_items: int,
    cfg: TrainConfig,
    purchase_only: bool,
    use_full_softmax: bool = False,
) -> Dict[str, float]:
    model.eval()
    dataset = SASRecDataset(
        split_data,
        max_len=cfg.max_len,
        purchase_only=purchase_only,
        n_items=n_items,
        n_negatives=0,
        seed=cfg.seed,
    )
    if len(dataset) == 0:
        return {"MRR": 0.0, "HR@10": 0.0, "NDCG@10": 0.0, "n": 0.0}

    loader = DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        collate_fn=collate_sasrec,
        num_workers=cfg.num_workers,
    )
    device = torch.device(cfg.device)
    all_ranks = []
    rng = np.random.default_rng(cfg.seed)

    for batch in loader:
        item_ids = batch["item_ids"].to(device)
        behavior_ids = batch["behavior_ids"].to(device)
        targets = batch["target_item"].to(device)

        if use_full_softmax or cfg.eval_negatives <= 0:
            scores = model.predict(item_ids, behavior_ids)
            ranks = rank_targets(scores, targets)
        else:
            batch_size = targets.size(0)
            cand = torch.zeros(batch_size, cfg.eval_negatives + 1, dtype=torch.long, device=device)
            cand[:, 0] = targets
            for i in range(batch_size):
                banned = set(item_ids[i].tolist()) | {int(targets[i].item()), 0}
                negs = []
                while len(negs) < cfg.eval_negatives:
                    neg = int(rng.integers(1, n_items + 1))
                    if neg not in banned and neg not in negs:
                        negs.append(neg)
                cand[i, 1:] = torch.tensor(negs, device=device)
            scores = model.predict(item_ids, behavior_ids, candidate_items=cand)
            pos_index = torch.zeros(batch_size, dtype=torch.long, device=device)
            ranks = rank_targets_among_candidates(scores, pos_index)
        all_ranks.append(ranks)

    return summarize_ranks(np.concatenate(all_ranks), ks=(5, 10))


def train_model(
    mode: str,
    sequences_payload: Dict,
    splits: Dict[str, Dict[int, object]],
    cfg: TrainConfig,
    output_dir: Path | str,
) -> Dict[str, object]:
    set_seed(cfg.seed)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    device = torch.device(cfg.device)

    n_items = int(sequences_payload["n_items"])
    purchase_only = mode == "purchase_only"
    model = build_model(mode, n_items, cfg).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

    train_ds = SASRecDataset(
        splits["train"],
        max_len=cfg.max_len,
        purchase_only=purchase_only,
        n_items=n_items,
        n_negatives=cfg.n_negatives,
        seed=cfg.seed,
    )
    train_loader = DataLoader(
        train_ds,
        batch_size=cfg.batch_size,
        shuffle=True,
        collate_fn=collate_sasrec,
        num_workers=cfg.num_workers,
    )

    history = []
    best_ndcg = -1.0
    best_path = output_dir / f"{mode}_best.pt"
    patience_left = cfg.patience

    for epoch in range(1, cfg.epochs + 1):
        model.train()
        losses = []
        for batch in tqdm(train_loader, desc=f"{mode} epoch {epoch}", leave=False):
            item_ids = batch["item_ids"].to(device)
            behavior_ids = batch["behavior_ids"].to(device)
            pos = batch["target_item"].to(device)
            neg = batch["neg_items"].to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = bce_ranking_loss(model, item_ids, behavior_ids, pos, neg)
            loss.backward()
            optimizer.step()
            losses.append(float(loss.item()))

        val_metrics = evaluate(
            model,
            splits["val"] if splits["val"] else splits["test"],
            n_items=n_items,
            cfg=cfg,
            purchase_only=purchase_only,
        )
        test_metrics = evaluate(
            model,
            splits["test"],
            n_items=n_items,
            cfg=cfg,
            purchase_only=purchase_only,
        )
        row = {
            "epoch": epoch,
            "loss": float(np.mean(losses)) if losses else None,
            "val": val_metrics,
            "test": test_metrics,
        }
        history.append(row)
        print(
            f"[{mode}] epoch={epoch} loss={row['loss']:.4f} "
            f"val({format_metrics(val_metrics)}) test({format_metrics(test_metrics)})"
        )

        ndcg = val_metrics.get("NDCG@10", 0.0)
        if ndcg > best_ndcg:
            best_ndcg = ndcg
            patience_left = cfg.patience
            torch.save(
                {
                    "model_state": model.state_dict(),
                    "mode": mode,
                    "n_items": n_items,
                    "config": asdict(cfg),
                    "val_metrics": val_metrics,
                    "test_metrics": test_metrics,
                },
                best_path,
            )
        else:
            patience_left -= 1
            if patience_left <= 0:
                print(f"[{mode}] early stopping at epoch {epoch}")
                break

    ckpt = torch.load(best_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    final_test = evaluate(model, splits["test"], n_items, cfg, purchase_only)
    return {
        "mode": mode,
        "best_checkpoint": str(best_path),
        "history": history,
        "best_val": ckpt.get("val_metrics"),
        "test_metrics": final_test,
        "config": asdict(cfg),
    }

print("Model + train helpers ready")

## 4. Load events

In [ ]:
INCLUDE = ("product_buy", "add_to_cart", "remove_from_cart")

if USE_SYNTHETIC:
    events = make_synthetic_events(n_users=300, n_items=150, seed=SEED)
    print("Synthetic events generated")
else:
    data_candidates = [
        DATA_DIR,
        Path("data/subsampled"),
        Path("../data/subsampled"),
        Path("Desktop/DS5500/data/subsampled"),
        Path.home() / "Desktop/DS5500/data/subsampled",
    ]
    data_dir = next((p for p in data_candidates if p.exists()), None)
    if data_dir is None:
        raise FileNotFoundError(
            "No subsampled data found. Run DataIngestion_Subsample.ipynb first, "
            "or set USE_SYNTHETIC=True."
        )
    print("Loading from:", data_dir.resolve())
    events = load_behavior_events(data_dir, include_behaviors=INCLUDE)

print(f"Rows: {len(events):,}")
display(events["behavior"].value_counts())
events.head()

## 5. Build user sequences + temporal split

Per user with ≥2 purchases: last purchase → test target; second-to-last → val (if available); earlier purchases → train instances. History includes events **before** the target purchase (cart signal allowed, no target leakage).

In [ ]:
payload = build_user_sequences(
    events,
    include_behaviors=INCLUDE,
    min_buys=MIN_BUYS,
    max_len=None,
    top_n_items=None if USE_SYNTHETIC else TOP_N_ITEMS,
)
print(f"Users={payload['n_users']:,} | items={payload['n_items']:,} | sequences={len(payload['sequences']):,}")

splits = temporal_split_sequences(payload["sequences"])
print(
    f"Split users — train: {len(splits['train']):,} | "
    f"val: {len(splits['val']):,} | test: {len(splits['test']):,}"
)

## 6. Train config

In [ ]:
cfg = TrainConfig(
    hidden_size=HIDDEN_SIZE,
    max_len=MAX_LEN,
    n_blocks=N_BLOCKS,
    n_heads=N_HEADS,
    dropout=DROPOUT,
    batch_size=BATCH_SIZE,
    lr=LR,
    epochs=EPOCHS,
    n_negatives=N_NEGATIVES,
    eval_negatives=EVAL_NEGATIVES,
    seed=SEED,
    device=DEVICE,
    patience=PATIENCE,
)
cfg

## 7. Train models

`behavior_aware` uses buy + add-to-cart + remove-from-cart in history.  
`purchase_only` strips cart events but uses the **same purchase targets**.

In [ ]:
modes = ["purchase_only", "behavior_aware"] if MODE == "both" else [MODE]
results = {}

for mode in modes:
    print(f"\n=== Training {mode} ===")
    result = train_model(mode, payload, splits, cfg, OUTPUT_DIR / mode)
    results[mode] = {
        "test_metrics": result["test_metrics"],
        "best_val": result["best_val"],
        "best_checkpoint": result["best_checkpoint"],
    }
    print(f"[{mode}] final test:", format_metrics(result["test_metrics"]))

results

## 8. Compare results

In [ ]:
rows = []
for mode, payload_res in results.items():
    m = payload_res["test_metrics"]
    rows.append({
        "model": mode,
        "HR@10": m.get("HR@10"),
        "NDCG@10": m.get("NDCG@10"),
        "MRR": m.get("MRR"),
        "HR@5": m.get("HR@5"),
        "NDCG@5": m.get("NDCG@5"),
        "n": int(m.get("n", 0)),
    })

comparison = pd.DataFrame(rows).set_index("model")
comparison

In [ ]:
summary_path = OUTPUT_DIR / "summary.json"
with summary_path.open("w") as f:
    json.dump(results, f, indent=2)
print("Wrote", summary_path.resolve())

## Notes

- Synthetic metrics are only for pipeline checking; report real-cohort numbers for the project.
- Keep the same `TOP_N_ITEMS` for both models so the comparison stays fair.
- Checkpoints: `artifacts/.../<mode>/<mode>_best.pt`